# Multiscale links (cross-pyramid-level edges)

**Geometry type:** `graph` · **Schema version:** `0.8`

This notebook is a deep dive into the **multiscale links layout** — how graph edges, cross-chunk links, and their attributes are organised across pyramid levels and how you can read each piece directly.

In v0.8, the cross-chunk-link family on disk is a set of **K-separated sharded vlen-bytes zarr arrays** (`cross_chunk_links/<delta>/kK`) rather than the pre-0.8 single monolithic blob.  Each record's `link_width` chunk-identities are recovered from the cell coord in the `kK` array; per-record cost drops from 64 bytes (sid_ndim=3, link_width=2) to 18 bytes.

Topics:

1. The on-disk layout: `links/<delta>/<chunk>` and `cross_chunk_links/<delta>/kK`
2. Write a small graph and build a 3-level pyramid
3. Inspect the directory tree — `0`, `+1`, `-1` arrays per level
4. Read intra-level edges (`delta=0`) — per-chunk row blobs
5. Read cross-level edges (`delta=+1`) — fine → coarse drill-up
6. Cross-chunk-link kN arrays: direct cell lookup + leaf enumeration
7. Per-link attributes, intra- *and* cross-chunk
8. Storage modes: `none` vs `implicit` vs `explicit`
9. Depth knob: `cross_level_depth=2`
10. Migration: rewrite a v0.7 store into the v0.8 layout in place
11. Validate

In [1]:
import numpy as np
import tempfile, os
from pathlib import Path

_tmpdir = tempfile.mkdtemp(prefix="zvf_multiscale_")
STORE = os.path.join(_tmpdir, "graph.zarrvectors")
print("Store:", STORE)

Store: /var/folders/l9/gyv08mks3ls6mfdhxmq9hvsr0000gp/T/zvf_multiscale_sdq9piir/graph.zarrvectors


## 1 · The on-disk layout

Under the 0.8 schema, every link-family path carries a `<level_delta>` segment that says how many pyramid levels the edges span:

```
/<N>/links/<delta>/<chunk_key>                     # per-chunk byte blob
/<N>/link_attributes/<name>/<delta>/<chunk_key>    # parallel attributes
/<N>/cross_chunk_links/<delta>/kK                  # K-separated sharded
                                                   #   vlen-bytes zarr arrays
                                                   #   (one kK per distinct K)
/<N>/cross_chunk_link_attributes/<name>/<delta>/kK
```

Convention for `<delta>`:

| Segment | Meaning |
|---------|---------|
| `0`     | intra-level (the default everywhere)        |
| `+1`    | this level to `this_level + 1` (one coarser) |
| `-1`    | this level to `this_level - 1` (one finer)   |
| `+N` / `-N` | jumps of N levels |

For the **cross-chunk-link** arrays specifically, `K` is the number of distinct chunks each record touches (`1 ≤ K ≤ link_width`).  A record whose sorted-unique chunks are `(c_0, …, c_{K-1})` lives at cell coord
`(c_0 - origin) ⧺ … ⧺ (c_{K-1} - origin)` of the `kK` array, where `origin = kK.attrs.chunk_origin`.

Endpoint identities inside one record are recovered from a per-endpoint `chunk_index` (`ci`) into that cell's K sorted chunks — no chunk coords appear in the payload.  Per-record encoding: `L * uint8 ci || L * int64 vi` = `9 * L` bytes.

Use the path helpers in `zarr_vectors.core.paths` to compose paths — never hard-code the `<delta>` formatting yourself.

In [2]:
from zarr_vectors.core.paths import (
    format_delta, parse_delta,
    links_path, cross_chunk_links_path,
    link_attributes_path, cross_chunk_link_attributes_path,
)

print("format_delta(0)  =", format_delta(0))
print("format_delta(+1) =", format_delta(1))
print("format_delta(-2) =", format_delta(-2))
print()
print("links_path(0)              =", links_path(0))
print("links_path(+1)             =", links_path(1))
print("cross_chunk_links_path(-1) =", cross_chunk_links_path(-1))
print("link_attributes_path('weight', +1)            =", link_attributes_path('weight', 1))
print("cross_chunk_link_attributes_path('weight', 0) =", cross_chunk_link_attributes_path('weight', 0))

format_delta(0)  = 0
format_delta(+1) = +1
format_delta(-2) = -2

links_path(0)              = links/0
links_path(+1)             = links/+1
cross_chunk_links_path(-1) = cross_chunk_links/-1
link_attributes_path('weight', +1)            = link_attributes/weight/+1
cross_chunk_link_attributes_path('weight', 0) = cross_chunk_link_attributes/weight/0


## 2 · Write a small graph and build a 3-level pyramid

We use a small (500-node) graph in a 400 µm cube so the pyramid produces a handful of metanodes per level — easy to eyeball.  Each level will roughly 8× coarsen the previous one.

Defaults pick up `cross_level_depth=1` and `cross_level_storage="explicit"` — we'll override those below to compare modes.  Default `explicit` writes both `+1` at the finer level and `-1` at the coarser level for every adjacent pair.

In [3]:
from zarr_vectors.types.graphs import write_graph
from zarr_vectors.multiresolution.coarsen import build_pyramid
from zarr_vectors.constants import XLEVEL_EXPLICIT, XLEVEL_IMPLICIT, XLEVEL_NONE

rng = np.random.default_rng(0)
N = 500
positions = rng.uniform(0.0, 400.0, size=(N, 3)).astype(np.float32)
edges = np.stack([np.arange(N - 1), np.arange(1, N)], axis=1).astype(np.int64)
edge_weights = rng.uniform(0.1, 1.0, size=len(edges)).astype(np.float32)

write_graph(
    STORE,
    positions=positions,
    edges=edges,
    object_ids=np.zeros(N, dtype=np.int64),
    chunk_shape=(100.0, 100.0, 100.0),
    bin_shape=(25.0, 25.0, 25.0),
    edge_attributes={"weight": edge_weights},
)

build_pyramid(
    STORE,
    factors=[(2.0, 1.0), (2.0, 1.0)],
    cross_level_depth=1,                # ±1 between every adjacent pair
    cross_level_storage=XLEVEL_EXPLICIT,  # store both +1 and -1
)
print("Build complete.")

/var/folders/l9/gyv08mks3ls6mfdhxmq9hvsr0000gp/T/ipykernel_30739/1986925154.py:11: DeprecationWarning: `edge_attributes` is deprecated; use `link_attributes`.
  write_graph(


Build complete.


## 3 · Walk the on-disk tree

Each resolution level should now carry `links/0` (intra-level edges) and, depending on its position in the pyramid, some combination of `links/+1`, `links/-1`, `cross_chunk_links/+1`, `cross_chunk_links/-1`.

In [4]:
from zarr_vectors.core.store import open_store, list_resolution_levels, get_resolution_level
from zarr_vectors.constants import LINKS, CROSS_CHUNK_LINKS
from zarr_vectors.core.arrays import list_link_deltas, list_cross_link_deltas

root = open_store(STORE)
levels = sorted(list_resolution_levels(root))
print(f"Pyramid levels: {levels}")
print()
for lvl in levels:
    lg = get_resolution_level(root, lvl)
    ld = list_link_deltas(lg)
    cd = list_cross_link_deltas(lg)
    print(f"  resolution_{lvl}: links/<delta> = {ld}    cross_chunk_links/<delta> = {cd}")

Pyramid levels: [0, 1, 2]

  resolution_0: links/<delta> = [0, 1]    cross_chunk_links/<delta> = [0]
  resolution_1: links/<delta> = [-1, 1]    cross_chunk_links/<delta> = [0]
  resolution_2: links/<delta> = [-1]    cross_chunk_links/<delta> = [0]


Reading this:

- Level 0 has `+1` (drill *up* to level 1) but no `-1` (nothing below).
- Mid levels carry both `+1` and `-1`.
- The top level has `-1` but no `+1` (nothing above).

If you peek at the actual directory you'll see one subdir per `<delta>`:

In [5]:
from pathlib import Path
level0_links = Path(STORE) / "0" / "links"
print(f"contents of {level0_links}:")
for child in sorted(level0_links.iterdir()):
    print(" ", child.name)

# Cross-chunk-link kN arrays for delta=+1 at level 0 (if any).
level0_ccl_plus1 = Path(STORE) / "0" / "cross_chunk_links" / "+1"
if level0_ccl_plus1.exists():
    print(f"\ncontents of {level0_ccl_plus1}:")
    for child in sorted(level0_ccl_plus1.iterdir()):
        print(" ", child.name)


contents of /var/folders/l9/gyv08mks3ls6mfdhxmq9hvsr0000gp/T/zvf_multiscale_sdq9piir/graph.zarrvectors/0/links:
  +1
  0
  zarr.json


## 4 · Intra-level edges — `delta=0` (unchanged behaviour)

Reading `delta=0` matches the pre-0.4 behaviour of `read_chunk_links`.  You get one list of `(M_k, 2)` arrays per spatial chunk; each row is a pair of local-vertex indices.

In [6]:
from zarr_vectors.core.arrays import read_chunk_links, list_chunk_keys

lg0 = get_resolution_level(root, 0)
chunk_keys = list_chunk_keys(lg0, LINKS + "/0")
print(f"level 0 has links/0 in {len(chunk_keys)} chunks")
for ck in chunk_keys[:3]:
    groups = read_chunk_links(lg0, ck, link_width=2, delta=0)
    n = sum(len(g) for g in groups)
    print(f"  chunk {ck}: {n} intra-chunk edges (groups: {len(groups)})")

level 0 has links/0 in 6 chunks
  chunk (1, 0, 1): 1 intra-chunk edges (groups: 1)
  chunk (1, 2, 2): 1 intra-chunk edges (groups: 1)
  chunk (2, 0, 3): 1 intra-chunk edges (groups: 1)


## 5 · Cross-level edges — `delta=+1` (drill up)

Cross-level edges are conceptually trivial: every fine vertex has one edge to its coarse parent metanode.  The build splits those edges into:

- **chunk-aligned** edges → `links/+1/<chunk_key>` when the source chunk key matches the coarse target chunk key;
- **cross-chunk** edges → `cross_chunk_links/+1/data` otherwise.

For a `links/+1` row, *column 0* is the local vertex index in the source chunk **at the owning level**; *column 1* is the local vertex index in the same chunk key **at level + 1**.

In [7]:
plus1_chunks = list_chunk_keys(lg0, LINKS + "/+1")
print(f"level 0 has links/+1 in {len(plus1_chunks)} chunks (chunk-aligned cross-level edges)")
total_plus1 = 0
for ck in plus1_chunks:
    g = read_chunk_links(lg0, ck, link_width=2, delta=1)
    n = sum(len(x) for x in g)
    total_plus1 += n
    if n:
        sample = g[0][:3]
        print(f"  chunk {ck}: {n} edges, sample rows (fine_local, coarse_local):")
        for row in sample:
            print(f"     {tuple(row)}")
print(f"\nTotal chunk-aligned +1 edges at level 0: {total_plus1}")

level 0 has links/+1 in 64 chunks (chunk-aligned cross-level edges)
  chunk (0, 0, 0): 6 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(0))
     (np.int64(1), np.int64(4))
     (np.int64(2), np.int64(0))
  chunk (0, 0, 1): 6 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(0))
     (np.int64(1), np.int64(2))
     (np.int64(2), np.int64(3))
  chunk (0, 0, 2): 6 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(1))
     (np.int64(1), np.int64(0))
     (np.int64(2), np.int64(4))
  chunk (0, 0, 3): 6 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(0))
     (np.int64(1), np.int64(2))
     (np.int64(2), np.int64(1))
  chunk (0, 1, 0): 7 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(0))
     (np.int64(1), np.int64(3))
     (np.int64(2), np.int64(1))
  chunk (0, 1, 1): 11 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(3))
     (np.in

  chunk (2, 2, 2): 9 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(1))
     (np.int64(1), np.int64(4))
     (np.int64(2), np.int64(3))
  chunk (2, 2, 3): 9 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(1))
     (np.int64(1), np.int64(2))
     (np.int64(2), np.int64(0))


  chunk (2, 3, 0): 10 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(0))
     (np.int64(1), np.int64(4))
     (np.int64(2), np.int64(3))
  chunk (2, 3, 1): 4 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(1))
     (np.int64(1), np.int64(0))
     (np.int64(2), np.int64(2))
  chunk (2, 3, 2): 6 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(3))
     (np.int64(1), np.int64(2))
     (np.int64(2), np.int64(1))


  chunk (2, 3, 3): 7 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(4))
     (np.int64(1), np.int64(5))
     (np.int64(2), np.int64(1))
  chunk (3, 0, 0): 13 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(1))
     (np.int64(1), np.int64(5))
     (np.int64(2), np.int64(2))
  chunk (3, 0, 1): 3 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(1))
     (np.int64(1), np.int64(0))
     (np.int64(2), np.int64(2))


  chunk (3, 0, 2): 10 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(1))
     (np.int64(1), np.int64(6))
     (np.int64(2), np.int64(5))
  chunk (3, 0, 3): 8 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(4))
     (np.int64(1), np.int64(3))
     (np.int64(2), np.int64(1))
  chunk (3, 1, 0): 9 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(3))
     (np.int64(1), np.int64(4))
     (np.int64(2), np.int64(4))
  chunk (3, 1, 1): 10 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(1))
     (np.int64(1), np.int64(5))
     (np.int64(2), np.int64(5))
  chunk (3, 1, 2): 10 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(3))
     (np.int64(1), np.int64(1))
     (np.int64(2), np.int64(0))
  chunk (3, 1, 3): 18 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(5))
     (np.int64(1), np.int64(2))
     (np.int64(2), np.int64(3))
  chunk (3, 

  chunk (3, 2, 3): 6 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(0))
     (np.int64(1), np.int64(2))
     (np.int64(2), np.int64(4))
  chunk (3, 3, 0): 10 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(4))
     (np.int64(1), np.int64(6))
     (np.int64(2), np.int64(3))
  chunk (3, 3, 1): 12 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(2))
     (np.int64(1), np.int64(4))
     (np.int64(2), np.int64(1))
  chunk (3, 3, 2): 7 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(4))
     (np.int64(1), np.int64(6))
     (np.int64(2), np.int64(0))
  chunk (3, 3, 3): 5 edges, sample rows (fine_local, coarse_local):
     (np.int64(0), np.int64(0))
     (np.int64(1), np.int64(1))
     (np.int64(2), np.int64(4))

Total chunk-aligned +1 edges at level 0: 500


## 6 · Cross-chunk + cross-level: `cross_chunk_links/+1/kK`

When a fine vertex's coarse parent lives in a *different* chunk grid cell, the edge can't be expressed by a per-chunk row — it goes into the K-separated sharded `kN` arrays under `cross_chunk_links/+1/`.

We can either:

- Read every record at this delta back as a flat list (legacy-shape API, useful for porting old code).
- Walk populated cells directly to inspect the on-disk shape.
- Look up records between two specific chunks in one sharded zarr cell read — that's the workload the new layout was built for.

In [8]:
from zarr_vectors.core.arrays import (
    read_cross_chunk_links,
    read_cross_chunk_link_leaf,
    list_cross_chunk_link_leaves,
)

# Whole-delta read (legacy-shape return).
ccl_plus1 = read_cross_chunk_links(lg0, delta=1)
print(f"level 0 has {len(ccl_plus1)} cross-chunk +1 records")
for a, b in ccl_plus1[:3]:
    (ca, la), (cb, lb) = a, b
    print(f"  endpoint 0: chunk={ca} vi={la}  ->  endpoint 1: chunk={cb} vi={lb}")

print()
print("populated cells (each row is the sorted-unique-chunks tuple for that cell):")
leaves = list_cross_chunk_link_leaves(lg0, delta=1)
for L in leaves[:5]:
    print(f"  {L}")

# Direct cell lookup for a single chunk pair.  Pick any pair from the
# enumeration above; the read goes through one sharded zarr cell.
if leaves:
    pair = leaves[0]
    cell = read_cross_chunk_link_leaf(lg0, pair, delta=1)
    print()
    print(f"direct cell lookup at sorted chunks {pair}: {len(cell)} records")
    for rec in cell[:3]:
        (ca, la), (cb, lb) = rec
        print(f"  ({ca}, {la}) <-> ({cb}, {lb})")

level 0 has 0 cross-chunk +1 records

populated cells (each row is the sorted-unique-chunks tuple for that cell):


## 7 · Per-link attributes — intra- and cross-chunk

Two parallel attribute namespaces exist:

- `link_attributes/<name>/<delta>/<chunk_key>` — parallel to `links/<delta>/<chunk_key>`, one ragged group per spatial chunk.
- `cross_chunk_link_attributes/<name>/<delta>/data` — *new in 0.4*, parallel to `cross_chunk_links/<delta>/data`; one flat row per cross-chunk link in the same order.

The build wrote `delta=0` attributes from the `edge_attributes={'weight': ...}` we passed into `write_graph`.  We'll also write a cross-chunk attribute by hand to show the new API.

Note: cross-chunk link attribute writes enforce `len(values) == num_links` at runtime — a misaligned write fails loudly instead of silently corrupting the parallel array.

In [9]:
from zarr_vectors.core.arrays import (
    create_cross_chunk_link_attributes_array,
    write_cross_chunk_link_attributes,
    read_cross_chunk_link_attributes,
)

# Re-open for writing.
root_rw = open_store(STORE, mode="r+")
lg0_rw = get_resolution_level(root_rw, 0)

# Cross-chunk +1 link attributes: one float per cross-chunk-link record.
# The writer slices ``weights`` per kN cell using the link arrays'
# existing record counts — pass the full flat array and a total
# ``num_links`` count, and v0.8 distributes them into matching cells.
num_ccl_plus1 = len(ccl_plus1)
if num_ccl_plus1:
    create_cross_chunk_link_attributes_array(lg0_rw, "weight", dtype="float32", delta=1)
    weights = np.linspace(0.0, 1.0, num_ccl_plus1, dtype=np.float32)
    write_cross_chunk_link_attributes(
        lg0_rw, "weight", weights, num_links=num_ccl_plus1, delta=1,
    )
    back = read_cross_chunk_link_attributes(lg0_rw, "weight", delta=1)
    print(f"wrote/read {len(back)} cross-chunk-link weights at delta=+1")
    print(f"first 5: {back[:5]}")
else:
    print("no cross-chunk +1 records at level 0; skipping attribute round-trip")

no cross-chunk +1 records at level 0; skipping attribute round-trip


Length-invariant check (this *should* raise):

In [10]:
from zarr_vectors.exceptions import ArrayError

if num_ccl_plus1:
    try:
        bad = np.zeros(num_ccl_plus1 + 7, dtype=np.float32)
        write_cross_chunk_link_attributes(
            lg0_rw, "weight", bad, num_links=num_ccl_plus1, delta=1,
        )
    except ArrayError as e:
        print(f"ArrayError raised as expected:\n  {e}")

## 8 · Storage modes: `explicit` vs `implicit` vs `none`

Three modes control whether `-N` arrays are materialised at all:

| Mode | Writes `+N` at fine level? | Writes `-N` at coarse level? |
|------|----------------------------|------------------------------|
| `none`     | no  | no  |
| `implicit` | yes | no  |
| `explicit` | yes | yes |

`implicit` saves storage; the `-N` direction is reconstructed by reading the `+N` array at the target level and swapping endpoints.  `explicit` materialises both, paying disk for O(1) reads in both directions.

Whenever any `cross_chunk_links/<delta>/` group exists in a store, the root `format_capabilities` carries both `multiscale_links` and `partitioned_cross_chunk_links` — readers refuse to open a store with the former but not the latter (those are pre-0.8 stores; see section 10 for the in-place migration).

Let's rebuild against fresh stores to compare on-disk footprints.

In [11]:
import shutil

def _build_one(name, *, depth, storage):
    path = os.path.join(_tmpdir, f"{name}.zarrvectors")
    if os.path.exists(path):
        shutil.rmtree(path)
    write_graph(
        path,
        positions=positions,
        edges=edges,
        object_ids=np.zeros(N, dtype=np.int64),
        chunk_shape=(100.0, 100.0, 100.0),
        bin_shape=(25.0, 25.0, 25.0),
        edge_attributes={"weight": edge_weights},
    )
    build_pyramid(
        path, factors=[(2.0, 1.0), (2.0, 1.0)],
        cross_level_depth=depth, cross_level_storage=storage,
    )
    return path

def _scan(path):
    r = open_store(path)
    out = []
    for lvl in sorted(list_resolution_levels(r)):
        lg = get_resolution_level(r, lvl)
        out.append((lvl, list_link_deltas(lg), list_cross_link_deltas(lg)))
    return out

for mode in (XLEVEL_NONE, XLEVEL_IMPLICIT, XLEVEL_EXPLICIT):
    p = _build_one(f"graph_{mode}", depth=1, storage=mode)
    print(f"\n--- cross_level_storage = {mode!r} ---")
    for lvl, ld, cd in _scan(p):
        print(f"  resolution_{lvl}: links/<delta>={ld}  cross_chunk_links/<delta>={cd}")

/var/folders/l9/gyv08mks3ls6mfdhxmq9hvsr0000gp/T/ipykernel_30739/4115966381.py:7: DeprecationWarning: `edge_attributes` is deprecated; use `link_attributes`.
  write_graph(



--- cross_level_storage = 'none' ---
  resolution_0: links/<delta>=[]  cross_chunk_links/<delta>=[0]
  resolution_1: links/<delta>=[]  cross_chunk_links/<delta>=[0]
  resolution_2: links/<delta>=[]  cross_chunk_links/<delta>=[0]


/var/folders/l9/gyv08mks3ls6mfdhxmq9hvsr0000gp/T/ipykernel_30739/4115966381.py:7: DeprecationWarning: `edge_attributes` is deprecated; use `link_attributes`.
  write_graph(



--- cross_level_storage = 'implicit' ---
  resolution_0: links/<delta>=[0, 1]  cross_chunk_links/<delta>=[0]
  resolution_1: links/<delta>=[1]  cross_chunk_links/<delta>=[0]
  resolution_2: links/<delta>=[]  cross_chunk_links/<delta>=[0]


/var/folders/l9/gyv08mks3ls6mfdhxmq9hvsr0000gp/T/ipykernel_30739/4115966381.py:7: DeprecationWarning: `edge_attributes` is deprecated; use `link_attributes`.
  write_graph(



--- cross_level_storage = 'explicit' ---
  resolution_0: links/<delta>=[0, 1]  cross_chunk_links/<delta>=[0]
  resolution_1: links/<delta>=[-1, 1]  cross_chunk_links/<delta>=[0]
  resolution_2: links/<delta>=[-1]  cross_chunk_links/<delta>=[0]


## 9 · Depth knob: `cross_level_depth=2`

`cross_level_depth` controls how far the cross-level emission reaches:

- `0` — disabled (same as `storage="none"`).
- `N` — materialise up to `±N` for every adjacent pair we can reach.
- `-1` — walk *all* available pyramid levels.

At `depth=2` the writer composes the fine→parent map across two coarsening steps (`grandparent[i] = parent_at_L1[parent_at_L0[i]]`) so a single edge goes from a level-0 vertex straight to its level-2 metanode.

In [12]:
p2 = _build_one("graph_depth2", depth=2, storage=XLEVEL_EXPLICIT)
print("depth=2, explicit:")
for lvl, ld, cd in _scan(p2):
    print(f"  resolution_{lvl}: links/<delta>={ld}  cross_chunk_links/<delta>={cd}")

/var/folders/l9/gyv08mks3ls6mfdhxmq9hvsr0000gp/T/ipykernel_30739/4115966381.py:7: DeprecationWarning: `edge_attributes` is deprecated; use `link_attributes`.
  write_graph(


depth=2, explicit:
  resolution_0: links/<delta>=[0, 1, 2]  cross_chunk_links/<delta>=[0]
  resolution_1: links/<delta>=[-1, 1]  cross_chunk_links/<delta>=[0]
  resolution_2: links/<delta>=[-2, -1]  cross_chunk_links/<delta>=[0]


Expected (with a 3-level pyramid):

- Level 0 → `+1`, `+2`
- Level 1 → `-1`, `+1`
- Level 2 → `-1`, `-2`

Plus `0` everywhere from the original `write_graph` call.

## 10 · Migration from v0.7

If you still have a store written by zarr-vectors 0.7 (single monolithic `cross_chunk_links/<delta>/data` int64 blob), the v0.8 reader will refuse to open it with a pointer to the in-place migration helper.  Run it once per store:

```python
from zarr_vectors.migration import partition_legacy_cross_chunk_links

summary = partition_legacy_cross_chunk_links(store_path, dry_run=True)
# inspect summary['level_results'] before committing
partition_legacy_cross_chunk_links(store_path)
```

The helper:

1. Walks every level and every `<delta>` under each level.
2. Decodes records from the legacy int64 blob using the pre-0.8 endpoint encoding.
3. Re-emits records via the new writer, which lands them in the matching `kK` sharded arrays (with `ci = [0, 1]` canonicalization applied for `delta=0 L=2`).
4. Reorders parallel attribute rows by the canonical (K, lex(chunks)) cell walk and writes them into matching `kN` attribute arrays.
5. Deletes the legacy `data` blobs and stamps `partitioned_cross_chunk_links` on root `format_capabilities`, bumping `zv_version` to `"0.8.0"`.

The migration is idempotent (a second invocation no-ops thanks to the capability guard) and supports `dry_run=True` for a pre-flight report.

## 11 · Validate

The validator walks each `<delta>` subdir under `links/` and `cross_chunk_links/` and checks that endpoint chunk keys are present in the level's chunk grid.

In [13]:
from zarr_vectors.validate import validate

rv = validate(STORE, level=3)
print(rv.summary())

/Users/forrestc/ConnectomeStack/zarr-vectors-py/zarr_vectors/core/arrays.py:2811: RuntimeWarning: coroutine '_walk_populated_shards.<locals>._gather' was never awaited
  return []


Level 3 validation: PASS
  43 passed, 0 warnings, 0 errors


## Summary

| Concept | API |
|---------|-----|
| Pyramid with cross-level edges | `build_pyramid(path, cross_level_depth=N, cross_level_storage="explicit")` |
| Compose paths | `links_path(delta)`, `cross_chunk_links_path(delta)`, `link_attributes_path(name, delta)`, `cross_chunk_link_attributes_path(name, delta)` |
| List deltas on disk | `list_link_deltas(level)`, `list_cross_link_deltas(level)` |
| Read intra-level edges | `read_chunk_links(level, chunk, delta=0)` |
| Read cross-level edges | `read_chunk_links(level, chunk, delta=+1)` |
| Read cross-chunk links | `read_cross_chunk_links(level, delta=±N)` |
| Write/read cross-chunk-link attrs | `write_cross_chunk_link_attributes(level, name, values, num_links, delta)` / `read_cross_chunk_link_attributes(level, name, delta)` |

Endpoint convention recap:

- `links/<delta>/<chunk>` rows: column 0 = source-level local index, column 1 = local index in the **same chunk key** at level `+delta`.
- `cross_chunk_links/<delta>/data` rows: `((src_chunk, src_local), (tgt_chunk, tgt_local))` — `src_*` at the owning level, `tgt_*` at level `+delta`.

See `docs/multiscale-links.md` (or the plan notes in the repo) for the design rationale and the schema-0.4 breaking change details.

Under v0.8 the cross-chunk-link family is partitioned into K-separated sharded `kN` arrays per delta.  The high-level read API (`read_cross_chunk_links`) preserves the legacy tuple-shape; new direct-access helpers (`read_cross_chunk_link_leaf`, `list_cross_chunk_link_leaves`) expose the cell layout.  Stores authored by the v0.8 writer always advertise both `multiscale_links` and `partitioned_cross_chunk_links` capabilities; older v0.7 stores migrate in place via `zarr_vectors.migration.partition_legacy_cross_chunk_links`.
